# 07 - Batch and Realtime Inference

**Objectif :** charger le bundle final, réaliser une inférence batch sur le test untouched, puis simuler des prédictions temps réel sans API.

In [1]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

Project root: /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab
Environment: development


## 1. Load external holdout

In [2]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

holdout_df = CsvLoanDataLoader(path=settings.raw_test_path).load()
holdout_df.head()

2026-09-07 11:44:33 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab/data/raw/test.csv
2026-09-07 11:44:33 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (4500 lignes, 14 colonnes)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,35.0,male,Master,157292.0,15,MORTGAGE,9000.0,DEBTCONSOLIDATION,12.73,0.06,8.0,630,Yes,0
1,44.0,male,Master,82657.0,21,RENT,5150.0,EDUCATION,7.90,0.06,14.0,671,Yes,0
2,23.0,female,High School,31975.0,1,MORTGAGE,7024.0,DEBTCONSOLIDATION,13.16,0.22,4.0,673,Yes,0
3,24.0,male,High School,44116.0,3,OWN,8000.0,PERSONAL,10.65,0.18,2.0,593,Yes,0
4,22.0,female,Bachelor,42795.0,0,RENT,5000.0,MEDICAL,11.01,0.12,4.0,624,No,0


## 2. Load model bundle

In [3]:
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository

repository = JoblibModelBundleRepository()
bundle = repository.load(settings.model_bundle_path)
bundle["metadata"]

{'created_at_utc': '2026-09-07T08:07:52.496864+00:00',
 'runtime_versions': {'python': '3.14.1',
  'pandas': '2.3.3',
  'scikit-learn': '1.7.2',
  'xgboost': '2.1.4',
  'catboost': '1.2.10',
  'lightgbm': '4.6.0',
  'joblib': '1.5.2'},
 'feature_schema': ['person_age',
  'person_education',
  'person_income',
  'person_emp_exp',
  'person_home_ownership',
  'loan_amnt',
  'loan_intent',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'credit_score',
  'previous_loan_defaults_on_file',
  'dti',
  'log_income',
  'estimated_monthly_interest',
  'interest_to_income',
  'income_per_experience_year',
  'credit_score_band',
  'rate_per_score_point',
  'age_group',
  'exp_to_age',
  'amnt_int_ratio',
  'loan_risk_score',
  'loan_intent_risk',
  'credit_hist_to_age',
  'credit_hist_category',
  'age_first_credit',
  'emp_credit_hist_gap',
  'has_default_before',
  'risky_default_score',
  'home_risk',
  'edu_level',
  'risk_flags_count',
  'income_interest_interac

## 3. Configure raw scorer

In [4]:
from credit_risk_lab.application import RawLoanScorer

scorer = RawLoanScorer(bundle, threshold=settings.decision_threshold)

{
    "model_name": bundle["metadata"].get("model_name"),
    "serving_threshold": scorer.model_scorer.threshold,
    "test_rows_available": len(holdout_df),
}

{'model_name': 'CatBoost',
 'serving_threshold': 0.25,
 'test_rows_available': 4500}

## 4. Batch inference on first test rows

In [5]:
from credit_risk_lab.application import BatchInferenceRunner

batch_limit = 100
submission_path = settings.reports_dir / "submission.csv"

batch_runner = BatchInferenceRunner(scorer)
batch_result = batch_runner.predict(
    holdout_df,
    output_path=submission_path,
    limit=batch_limit,
)

print(f"Batch rows scored: {len(batch_result.submission)}")
print(f"Submission saved to: {batch_result.output_path}")

display(batch_result.submission.head(10))

2026-09-07 11:44:36 | INFO     | batch_inference | credit_risk_lab.application.scoring:predict:142 - Starting batch inference on 100 rows
2026-09-07 11:44:36 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:110 - [FeatureEngineering] Entrée - shape = (100, 13)
2026-09-07 11:44:36 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:110 - [FeatureEngineering] Sortie - shape = (100, 38)
2026-09-07 11:44:36 | INFO     | batch_inference | credit_risk_lab.application.scoring:predict:149 - Batch inference file saved to /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab/reports/submission.csv
Batch rows scored: 100
Submission saved to: /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab/reports/submission.csv


,request_id,source_row_index,model_name,threshold,probability_of_risk,risk_decision,risk_label,risk_band,person_age,person_income,loan_amnt,loan_int_rate,loan_percent_income,loan_intent,actual_label,is_correct
0,batch-000000,0,CatBoost,0.25,0.000001,0,low_risk,low,35.0,157292.0,9000.0,12.73,0.06,DEBTCONSOLIDATION,0,True
1,batch-000001,1,CatBoost,0.25,0.000001,0,low_risk,low,44.0,82657.0,5150.0,7.90,0.06,EDUCATION,0,True
2,batch-000002,2,CatBoost,0.25,0.000001,0,low_risk,low,23.0,31975.0,7024.0,13.16,0.22,DEBTCONSOLIDATION,0,True
3,batch-000003,3,CatBoost,0.25,0.000000,0,low_risk,low,24.0,44116.0,8000.0,10.65,0.18,PERSONAL,0,True
4,batch-000004,4,CatBoost,0.25,0.325817,1,high_risk,medium,22.0,42795.0,5000.0,11.01,0.12,MEDICAL,0,False
5,batch-000005,5,CatBoost,0.25,0.000000,0,low_risk,low,25.0,145342.0,20000.0,11.14,0.14,HOMEIMPROVEMENT,0,True
6,batch-000006,6,CatBoost,0.25,0.991505,1,high_risk,critical,28.0,34991.0,8700.0,14.61,0.25,VENTURE,1,True
7,batch-000007,7,CatBoost,0.25,0.000001,0,low_risk,low,24.0,112475.0,3000.0,7.50,0.03,DEBTCONSOLIDATION,0,True
8,batch-000008,8,CatBoost,0.25,0.999590,1,high_risk,critical,24.0,44701.0,13000.0,17.27,0.29,EDUCATION,1,True
9,batch-000009,9,CatBoost,0.25,0.000000,0,low_risk,low,23.0,57434.0,5000.0,6.99,0.09,PERSONAL,0,True


## 5. Batch inference risk distribution

In [6]:
px.histogram(
    batch_result.submission,
    x="probability_of_risk",
    color="risk_label",
    nbins=30,
    title="Batch inference - predicted risk probabilities",
    template="plotly_white",
).show()

## 6. Realtime inference simulation without API

In [7]:
from credit_risk_lab.application import RealtimeInferenceSimulator

realtime_limit = 5
simulator = RealtimeInferenceSimulator(
    scorer,
    min_pause_seconds=1,
    max_pause_seconds=5,
)

realtime_events = simulator.stream(
    holdout_df,
    limit=realtime_limit,
    request_prefix="loan-request",
    print_events=True,
)

display(realtime_events)

2026-09-07 11:44:39 | INFO     | realtime_inference | credit_risk_lab.application.scoring:stream:233 - Starting realtime inference simulation on 5 requests
2026-09-07 11:44:39 | INFO     | realtime_inference | credit_risk_lab.application.scoring:stream:241 - Receiving request loan-request-000001
2026-09-07 11:44:39 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:110 - [FeatureEngineering] Entrée - shape = (1, 13)
2026-09-07 11:44:39 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:110 - [FeatureEngineering] Sortie - shape = (1, 38)
2026-09-07 11:44:39 | INFO     | realtime_inference | credit_risk_lab.application.scoring:stream:260 - loan-request-000001 | probability=0.0000 | decision=0 | threshold=0.250 | next_pause=4.04s
loan-request-000001 | probability=0.0000 | decision=0 | threshold=0.250 | next_pause=4.04s
2026-09-07 11:44:43 | INFO     | realtime_infe

,request_id,row_position,probability_of_risk,risk_decision,threshold,model_name,pause_seconds
0,loan-request-000001,1,6.267403e-07,0,0.25,CatBoost,4.037
1,loan-request-000002,2,7.096417e-07,0,0.25,CatBoost,2.170
2,loan-request-000003,3,1.323309e-06,0,0.25,CatBoost,1.354
3,loan-request-000004,4,5.028438e-08,0,0.25,CatBoost,3.341
4,loan-request-000005,5,3.258172e-01,1,0.25,CatBoost,1.759


## 7. Realtime decision timeline

In [8]:
fig = px.line(
    realtime_events,
    x="request_id",
    y="probability_of_risk",
    markers=True,
    title="Realtime simulation - request-level risk score",
    template="plotly_white",
)
fig.add_hline(
    y=settings.decision_threshold,
    line_dash="dash",
    line_color="black",
    annotation_text=f"threshold={settings.decision_threshold:.3f}",
)
fig.show()